# EquiNexus SWEFil YOLO warm-start

Pretrain YOLO11-seg on the separately released SWEFil CC-BY-4.0 filament masks, then use the resulting checkpoint to initialize later training on the competition's allowed training annotations. The attached private Kaggle input contains only the audited sample: all calendar dates present in either the competition train or test data, exact competition observation names, raw duplicates, sunspot labels, and filament class targets were removed before upload. This notebook rechecks the filtered manifest and exact image overlap before use. No MAGFiLO labels outside the competition-provided training set are read. Final test inference remains limited to the supplied H-alpha images.

SWEFil source revision: `bc818c69a79266ad0fa91a539359485af57967cf`. Run configuration: 1024px, 30 epochs, batch size 4, patience 8, one Kaggle GPU. Outputs are written under `/kaggle/working/equinexus/`; the best checkpoint is the reusable artifact.


In [ ]:
import base64
from pathlib import Path
REPOSITORY_PATH = Path('/kaggle/working/EquiNexus')
SOURCE_COMMIT = 'd9e23b697796c600acd6b05d0a12362888388976'
SOURCE_BUNDLE_SHA256 = 'ae2dea578011c3f5fffe242ba314a440e5407a6f8a95b3a333bbd5810e0a467b'
SOURCE_FILES = {"src/filament_baseline/__init__.py":"IiIiU2NvcmVyLWFsaWduZWQgdmFsaWRhdGlvbiBiYXNlbGluZSBmb3IgRXF1aU5leHVzLiIiIgoKX192ZXJzaW9uX18gPSAiMC4xLjAiCg==","src/filament_baseline/data.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IHJlCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IGRlZmF1bHRkaWN0CmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKQU5OT1RBVElPTl9GSUxFTkFNRSA9ICJNQUdGaUxPXzEuMF9Bbm5vdGF0aW9uc19rYWdnbGUyMDI2X3RyYWluLmpzb24iCklNQUdFX1NJWkUgPSAyMDQ4CgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgSW1hZ2VWaWV3OgogICAgaW1hZ2VfaWQ6IHN0cgogICAgZmlsZV9uYW1lOiBzdHIKICAgIHdpZHRoOiBpbnQKICAgIGhlaWdodDogaW50CiAgICB5ZWFyOiBpbnQKICAgIHNpdGU6IHN0cgoKCkBkYXRhY2xhc3MoZnJvemVuPVRydWUpCmNsYXNzIENvbXBldGl0aW9uRGF0YXNldDoKICAgIHJvb3Q6IFBhdGgKICAgIHRyYWluX2ltYWdlX2RpcjogUGF0aAogICAgdGVzdF9pbWFnZV9kaXI6IFBhdGgKICAgIGFubm90YXRpb25fcGF0aDogUGF0aAogICAgaW1hZ2Vfdmlld3M6IHR1cGxlW0ltYWdlVmlldywgLi4uXQogICAgYW5ub3RhdGlvbnNfYnlfaW1hZ2VfaWQ6IGRpY3Rbc3RyLCB0dXBsZVtkaWN0W3N0ciwgQW55XSwgLi4uXV0KICAgIHZpZXdzX2J5X29ic2VydmF0aW9uOiBkaWN0W3N0ciwgdHVwbGVbSW1hZ2VWaWV3LCAuLi5dXQogICAgY2F0ZWdvcmllczogdHVwbGVbZGljdFtzdHIsIEFueV0sIC4uLl0KICAgIGltYWdlX3Jlc2l6ZV90bzogdHVwbGVbaW50LCBpbnRdIHwgTm9uZSA9IE5vbmUKCiAgICBAcHJvcGVydHkKICAgIGRlZiBvYnNlcnZhdGlvbl9uYW1lcyhzZWxmKSAtPiB0dXBsZVtzdHIsIC4uLl06CiAgICAgICAgcmV0dXJuIHR1cGxlKHNvcnRlZChzZWxmLnZpZXdzX2J5X29ic2VydmF0aW9uKSkKCiAgICBAcHJvcGVydHkKICAgIGRlZiB0ZXN0X2ltYWdlX2NvdW50KHNlbGYpIC0+IGludDoKICAgICAgICByZXR1cm4gc3VtKHBhdGguaXNfZmlsZSgpIGZvciBwYXRoIGluIHNlbGYudGVzdF9pbWFnZV9kaXIuaXRlcmRpcigpKQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIGFubm90YXRpb25fY291bnQoc2VsZikgLT4gaW50OgogICAgICAgIHJldHVybiBzdW0obGVuKGl0ZW1zKSBmb3IgaXRlbXMgaW4gc2VsZi5hbm5vdGF0aW9uc19ieV9pbWFnZV9pZC52YWx1ZXMoKSkKCgpkZWYgX2xvb2tzX2xpa2VfZGF0YXNldChyb290OiBQYXRoKSAtPiBib29sOgogICAgcmV0dXJuICgKICAgICAgICAocm9vdCAvICJ0cmFpbiIgLyBBTk5PVEFUSU9OX0ZJTEVOQU1FKS5pc19maWxlKCkKICAgICAgICBhbmQgKHJvb3QgLyAidHJhaW4iIC8gInRyYWluX2ltYWdlcyIpLmlzX2RpcigpCiAgICAgICAgYW5kIChyb290IC8gInRlc3QiIC8gInRlc3RfaW1hZ2VzIikuaXNfZGlyKCkKICAgICkKCgpkZWYgZmluZF9kYXRhc2V0X3Jvb3Qoc2VhcmNoX3Jvb3Q6IHN0ciB8IFBhdGgpIC0+IFBhdGg6CiAgICAiIiJGaW5kIHRoZSBjb21wZXRpdGlvbiBpbnB1dCBmb2xkZXIgd2l0aG91dCByZWx5aW5nIG9uIGEgZml4ZWQgS2FnZ2xlIG1vdW50IG5hbWUuIiIiCiAgICBzZWFyY2hfcm9vdCA9IFBhdGgoc2VhcmNoX3Jvb3QpLmV4cGFuZHVzZXIoKS5yZXNvbHZlKCkKICAgIGlmIG5vdCBzZWFyY2hfcm9vdC5leGlzdHMoKToKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIkRhdGEgc2VhcmNoIHBhdGggZG9lcyBub3QgZXhpc3Q6IHtzZWFyY2hfcm9vdH0iKQogICAgaWYgX2xvb2tzX2xpa2VfZGF0YXNldChzZWFyY2hfcm9vdCk6CiAgICAgICAgcmV0dXJuIHNlYXJjaF9yb290CgogICAgbWF0Y2hlcyA9IHNvcnRlZCgKICAgICAgICB7cGF0aC5wYXJlbnQucGFyZW50LnJlc29sdmUoKSBmb3IgcGF0aCBpbiBzZWFyY2hfcm9vdC5yZ2xvYihBTk5PVEFUSU9OX0ZJTEVOQU1FKX0KICAgICAgICBpZiBzZWFyY2hfcm9vdC5pc19kaXIoKQogICAgICAgIGVsc2Ugc2V0KCkKICAgICkKICAgIG1hdGNoZXMgPSBbcGF0aCBmb3IgcGF0aCBpbiBtYXRjaGVzIGlmIF9sb29rc19saWtlX2RhdGFzZXQocGF0aCldCiAgICBpZiBsZW4obWF0Y2hlcykgPT0gMToKICAgICAgICByZXR1cm4gbWF0Y2hlc1swXQogICAgaWYgbm90IG1hdGNoZXM6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoCiAgICAgICAgICAgIGYiQ291bGQgbm90IGZpbmQge0FOTk9UQVRJT05fRklMRU5BTUV9IHdpdGggdHJhaW4vdHJhaW5faW1hZ2VzIGFuZCAiCiAgICAgICAgICAgIGYidGVzdC90ZXN0X2ltYWdlcyBiZWxvdyB7c2VhcmNoX3Jvb3R9IgogICAgICAgICkKICAgIHJlbmRlcmVkID0gIlxuIi5qb2luKHN0cihwYXRoKSBmb3IgcGF0aCBpbiBtYXRjaGVzKQogICAgcmFpc2UgUnVudGltZUVycm9yKAogICAgICAgICJNb3JlIHRoYW4gb25lIGNvbXBldGl0aW9uIGRhdGFzZXQgd2FzIGZvdW5kOyBwYXNzIGFuIGV4cGxpY2l0IGRhdGFzZXQgcm9vdDpcbiIKICAgICAgICBmIntyZW5kZXJlZH0iCiAgICApCgoKZGVmIF95ZWFyX2FuZF9zaXRlKGZpbGVfbmFtZTogc3RyKSAtPiB0dXBsZVtpbnQsIHN0cl06CiAgICBzdGVtID0gUGF0aChmaWxlX25hbWUpLnN0ZW0KICAgIGlmIGxlbihzdGVtKSA8IDQgb3Igbm90IHN0ZW1bOjRdLmlzZGlnaXQoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiQ2Fubm90IGRlcml2ZSBjYXB0dXJlIHllYXIgZnJvbSBpbWFnZSBmaWxlbmFtZToge2ZpbGVfbmFtZX0iKQogICAgeWVhciA9IGludChzdGVtWzo0XSkKICAgIHNpdGVfbWF0Y2ggPSByZS5mdWxsbWF0Y2gociJcZHsxNH0oW0EtWmEtel17Mn0pIiwgc3RlbSkKICAgIHNpdGUgPSBzaXRlX21hdGNoLmdyb3VwKDEpIGlmIHNpdGVfbWF0Y2ggZWxzZSAidW5rbm93biIKICAgIHJldHVybiB5ZWFyLCBzaXRlCgoKZGVmIGxvYWRfZGF0YXNldChkYXRhX3Jvb3Q6IHN0ciB8IFBhdGgpIC0+IENvbXBldGl0aW9uRGF0YXNldDoKICAgIHJvb3QgPSBmaW5kX2RhdGFzZXRfcm9vdChkYXRhX3Jvb3QpCiAgICBhbm5vdGF0aW9uX3BhdGggPSByb290IC8gInRyYWluIiAvIEFOTk9UQVRJT05fRklMRU5BTUUKICAgIHRyYWluX2ltYWdlX2RpciA9IHJvb3QgLyAidHJhaW4iIC8gInRyYWluX2ltYWdlcyIKICAgIHRlc3RfaW1hZ2VfZGlyID0gcm9vdCAvICJ0ZXN0IiAvICJ0ZXN0X2ltYWdlcyIKICAgIHBheWxvYWQgPSBqc29uLmxvYWRzKGFubm90YXRpb25fcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCgogICAgaW1hZ2Vfdmlld3M6IGxpc3RbSW1hZ2VWaWV3XSA9IFtdCiAgICBpbWFnZV9pZHM6IHNldFtzdHJdID0gc2V0KCkKICAgIHZpZXdzX2J5X29ic2VydmF0aW9uOiBkaWN0W3N0ciwgbGlzdFtJbWFnZVZpZXddXSA9IGRlZmF1bHRkaWN0KGxpc3QpCiAgICBmb3IgcmVjb3JkIGluIHBheWxvYWQuZ2V0KCJpbWFnZXMiLCBbXSk6CiAgICAgICAgaW1hZ2VfaWQgPSBzdHIocmVjb3JkWyJpZCJdKQogICAgICAgIGZpbGVfbmFtZSA9IHN0cihyZWNvcmRbImZpbGVfbmFtZSJdKQogICAgICAgIGlmIFBhdGgoZmlsZV9uYW1lKS5uYW1lICE9IGZpbGVfbmFtZToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkV4cGVjdGVkIGEgYmFzZW5hbWUgaW4gQ09DTyBmaWxlX25hbWUsIGdvdDoge2ZpbGVfbmFtZX0iKQogICAgICAgIGlmIGltYWdlX2lkIGluIGltYWdlX2lkczoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkR1cGxpY2F0ZSBDT0NPIGltYWdlIGlkOiB7aW1hZ2VfaWR9IikKICAgICAgICBpbWFnZV9pZHMuYWRkKGltYWdlX2lkKQogICAgICAgIHllYXIsIHNpdGUgPSBfeWVhcl9hbmRfc2l0ZShmaWxlX25hbWUpCiAgICAgICAgdmlldyA9IEltYWdlVmlldygKICAgICAgICAgICAgaW1hZ2VfaWQ9aW1hZ2VfaWQsCiAgICAgICAgICAgIGZpbGVfbmFtZT1maWxlX25hbWUsCiAgICAgICAgICAgIHdpZHRoPWludChyZWNvcmRbIndpZHRoIl0pLAogICAgICAgICAgICBoZWlnaHQ9aW50KHJlY29yZFsiaGVpZ2h0Il0pLAogICAgICAgICAgICB5ZWFyPXllYXIsCiAgICAgICAgICAgIHNpdGU9c2l0ZSwKICAgICAgICApCiAgICAgICAgaWYgdmlldy53aWR0aCAhPSBJTUFHRV9TSVpFIG9yIHZpZXcuaGVpZ2h0ICE9IElNQUdFX1NJWkU6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgICAgICBmIkV4cGVjdGVkIHtJTUFHRV9TSVpFfXh7SU1BR0VfU0laRX0sIGdvdCB7dmlldy53aWR0aH14e3ZpZXcuaGVpZ2h0fSAiCiAgICAgICAgICAgICAgICBmImZvciB7ZmlsZV9uYW1lfSIKICAgICAgICAgICAgKQogICAgICAgIGltYWdlX3ZpZXdzLmFwcGVuZCh2aWV3KQogICAgICAgIHZpZXdzX2J5X29ic2VydmF0aW9uW2ZpbGVfbmFtZV0uYXBwZW5kKHZpZXcpCgogICAgYW5ub3RhdGlvbnM6IGRpY3Rbc3RyLCBsaXN0W2RpY3Rbc3RyLCBBbnldXV0gPSBkZWZhdWx0ZGljdChsaXN0KQogICAgZm9yIGFubm90YXRpb24gaW4gcGF5bG9hZC5nZXQoImFubm90YXRpb25zIiwgW10pOgogICAgICAgIGltYWdlX2lkID0gc3RyKGFubm90YXRpb25bImltYWdlX2lkIl0pCiAgICAgICAgaWYgaW1hZ2VfaWQgbm90IGluIGltYWdlX2lkczoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkFubm90YXRpb24gcmVmZXJlbmNlcyB1bmtub3duIGltYWdlIGlkOiB7aW1hZ2VfaWR9IikKICAgICAgICBpZiBpbnQoYW5ub3RhdGlvbi5nZXQoImlzY3Jvd2QiLCAwKSkgIT0gMDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVGhlIGNvbXBldGl0aW9uIHRyYWluaW5nIGFubm90YXRpb25zIHVuZXhwZWN0ZWRseSBjb250YWluIGNyb3dkIG1hc2tzIikKICAgICAgICBhbm5vdGF0aW9uc1tpbWFnZV9pZF0uYXBwZW5kKGFubm90YXRpb24pCgogICAgZm9yIGltYWdlX2lkIGluIGltYWdlX2lkczoKICAgICAgICBhbm5vdGF0aW9uc1tpbWFnZV9pZF0uc29ydChrZXk9bGFtYmRhIGl0ZW06IHN0cihpdGVtWyJpZCJdKSkKICAgICAgICBpZiBub3QgYW5ub3RhdGlvbnNbaW1hZ2VfaWRdOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiQW5ub3RhdG9yIHZpZXcge2ltYWdlX2lkfSBoYXMgbm8gZmlsYW1lbnQgYW5ub3RhdGlvbnMiKQoKICAgIGZvciBmaWxlX25hbWUsIHZpZXdzIGluIHZpZXdzX2J5X29ic2VydmF0aW9uLml0ZW1zKCk6CiAgICAgICAgZXhwZWN0ZWRfcGF0aCA9IHRyYWluX2ltYWdlX2RpciAvIGZpbGVfbmFtZQogICAgICAgIGlmIG5vdCBleHBlY3RlZF9wYXRoLmlzX2ZpbGUoKToKICAgICAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJDT0NPIGltYWdlIGZpbGUgaXMgbWlzc2luZzoge2V4cGVjdGVkX3BhdGh9IikKICAgICAgICB2aWV3cy5zb3J0KGtleT1sYW1iZGEgdmlldzogdmlldy5pbWFnZV9pZCkKCiAgICB0ZXN0X2ZpbGVzID0gW3BhdGggZm9yIHBhdGggaW4gdGVzdF9pbWFnZV9kaXIuaXRlcmRpcigpIGlmIHBhdGguaXNfZmlsZSgpXQogICAgaWYgbm90IHRlc3RfZmlsZXM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIk5vIHRlc3QgaW1hZ2VzIGZvdW5kIGluIHt0ZXN0X2ltYWdlX2Rpcn0iKQoKICAgIHJldHVybiBDb21wZXRpdGlvbkRhdGFzZXQoCiAgICAgICAgcm9vdD1yb290LAogICAgICAgIHRyYWluX2ltYWdlX2Rpcj10cmFpbl9pbWFnZV9kaXIsCiAgICAgICAgdGVzdF9pbWFnZV9kaXI9dGVzdF9pbWFnZV9kaXIsCiAgICAgICAgYW5ub3RhdGlvbl9wYXRoPWFubm90YXRpb25fcGF0aCwKICAgICAgICBpbWFnZV92aWV3cz10dXBsZShzb3J0ZWQoaW1hZ2Vfdmlld3MsIGtleT1sYW1iZGEgdmlldzogKHZpZXcuZmlsZV9uYW1lLCB2aWV3LmltYWdlX2lkKSkpLAogICAgICAgIGFubm90YXRpb25zX2J5X2ltYWdlX2lkPXsKICAgICAgICAgICAgaW1hZ2VfaWQ6IHR1cGxlKGl0ZW1zKSBmb3IgaW1hZ2VfaWQsIGl0ZW1zIGluIGFubm90YXRpb25zLml0ZW1zKCkKICAgICAgICB9LAogICAgICAgIHZpZXdzX2J5X29ic2VydmF0aW9uPXsKICAgICAgICAgICAgZmlsZV9uYW1lOiB0dXBsZSh2aWV3cykKICAgICAgICAgICAgZm9yIGZpbGVfbmFtZSwgdmlld3MgaW4gc29ydGVkKHZpZXdzX2J5X29ic2VydmF0aW9uLml0ZW1zKCkpCiAgICAgICAgfSwKICAgICAgICBjYXRlZ29yaWVzPXR1cGxlKHBheWxvYWQuZ2V0KCJjYXRlZ29yaWVzIiwgW10pKSwKICAgICkKCgpkZWYgaW52ZW50b3J5KGRhdGFzZXQ6IENvbXBldGl0aW9uRGF0YXNldCkgLT4gZGljdFtzdHIsIGludF06CiAgICByZXR1cm4gewogICAgICAgICJvYnNlcnZhdGlvbl9jb3VudCI6IGxlbihkYXRhc2V0LnZpZXdzX2J5X29ic2VydmF0aW9uKSwKICAgICAgICAiYW5ub3RhdG9yX3ZpZXdfY291bnQiOiBsZW4oZGF0YXNldC5pbWFnZV92aWV3cyksCiAgICAgICAgImFubm90YXRpb25fY291bnQiOiBkYXRhc2V0LmFubm90YXRpb25fY291bnQsCiAgICAgICAgInRyYWluX2ltYWdlX2ZpbGVfY291bnQiOiBzdW0oCiAgICAgICAgICAgIHBhdGguaXNfZmlsZSgpIGZvciBwYXRoIGluIGRhdGFzZXQudHJhaW5faW1hZ2VfZGlyLml0ZXJkaXIoKQogICAgICAgICksCiAgICAgICAgInRlc3RfaW1hZ2VfZmlsZV9jb3VudCI6IGRhdGFzZXQudGVzdF9pbWFnZV9jb3VudCwKICAgIH0K","src/filament_baseline/external_data.py":"...","src/filament_baseline/runtime.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGltcG9ydGxpYi5tZXRhZGF0YQppbXBvcnQgcGxhdGZvcm0KaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKClRSQUNLRURfUlVOVElNRV9QQUNLQUdFUyA9ICgKICAgICJudW1weSIsCiAgICAib3BlbmN2LXB5dGhvbi1oZWFkbGVzcyIsCiAgICAicGFuZGFzIiwKICAgICJzY2lraXQtbGVhcm4iLAogICAgInNjaXB5IiwKICAgICJtYXRwbG90bGliIiwKICAgICJweWNvY290b29scyIsCiAgICAidG9yY2giLAopCgoKZGVmIHBhY2thZ2VfdmVyc2lvbnMoKSAtPiBkaWN0W3N0ciwgc3RyIHwgTm9uZV06CiAgICB2ZXJzaW9uczogZGljdFtzdHIsIHN0ciB8IE5vbmVdID0ge30KICAgIGZvciBwYWNrYWdlIGluIFRSQUNLRURfUlVOVElNRV9QQUNLQUdFUzoKICAgICAgICB0cnk6CiAgICAgICAgICAgIHZlcnNpb25zW3BhY2thZ2VdID0gaW1wb3J0bGliLm1ldGFkYXRhLnZlcnNpb24ocGFja2FnZSkKICAgICAgICBleGNlcHQgaW1wb3J0bGliLm1ldGFkYXRhLlBhY2thZ2VOb3RGb3VuZEVycm9yOgogICAgICAgICAgICB2ZXJzaW9uc1twYWNrYWdlXSA9IE5vbmUKICAgIHJldHVybiB2ZXJzaW9ucwoKCmRlZiBoYXJkd2FyZV9pbmZvKCkgLT4gZGljdFtzdHIsIEFueV06CiAgICByZXN1bHQ6IGRpY3Rbc3RyLCBBbnldID0gewogICAgICAgICJwbGF0Zm9ybSI6IHBsYXRmb3JtLnBsYXRmb3JtKCksCiAgICAgICAgInB5dGhvbiI6IHN5cy52ZXJzaW9uLnNwbGl0KClbMF0sCiAgICAgICAgImN1ZGFfYXZhaWxhYmxlIjogRmFsc2UsCiAgICAgICAgInRvcmNoX3ZlcnNpb24iOiBOb25lLAogICAgICAgICJ0b3JjaF9jdWRhX3ZlcnNpb24iOiBOb25lLAogICAgICAgICJncHVfZGV2aWNlcyI6IFtdLAogICAgICAgICJudmlkaWFfc21pX3N1bW1hcnkiOiBOb25lLAogICAgfQogICAgdHJ5OgogICAgICAgIHJlc3VsdFsibnZpZGlhX3NtaV9zdW1tYXJ5Il0gPSBzdWJwcm9jZXNzLnJ1bigKICAgICAgICAgICAgWwogICAgICAgICAgICAgICAgIm52aWRpYS1zbWkiLAogICAgICAgICAgICAgICAgIi0tcXVlcnktZ3B1PW5hbWUsbWVtb3J5LnRvdGFsLGRyaXZlcl92ZXJzaW9uIiwKICAgICAgICAgICAgICAgICItLWZvcm1hdD1jc3Ysbm9oZWFkZXIiLAogICAgICAgICAgICBdLAogICAgICAgICAgICBjaGVjaz1UcnVlLAogICAgICAgICAgICBjYXB0dXJlX291dHB1dD1UcnVlLAogICAgICAgICAgICB0ZXh0PVRydWUsCiAgICAgICAgKS5zdGRvdXQuc3RyaXAoKS5zcGxpdGxpbmVzKCkKICAgIGV4Y2VwdCAoT1NFcnJvciwgc3VicHJvY2Vzcy5DYWxsZWRQcm9jZXNzRXJyb3IpOgogICAgICAgIHBhc3MKICAgIHRyeToKICAgICAgICBpbXBvcnQgdG9yY2gKICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgICAgICByZXR1cm4gcmVzdWx0CgogICAgcmVzdWx0WyJ0b3JjaF92ZXJzaW9uIl0gPSB0b3JjaC5fX3ZlcnNpb25fXwogICAgcmVzdWx0WyJ0b3JjaF9jdWRhX3ZlcnNpb24iXSA9IHRvcmNoLnZlcnNpb24uY3VkYQogICAgcmVzdWx0WyJjdWRhX2F2YWlsYWJsZSJdID0gYm9vbCh0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpKQogICAgaWYgcmVzdWx0WyJjdWRhX2F2YWlsYWJsZSJdOgogICAgICAgIGRldmljZXMgPSBbXQogICAgICAgIGZvciBpbmRleCBpbiByYW5nZSh0b3JjaC5jdWRhLmRldmljZV9jb3VudCgpKToKICAgICAgICAgICAgcHJvcHMgPSB0b3JjaC5jdWRhLmdldF9kZXZpY2VfcHJvcGVydGllcyhpbmRleCkKICAgICAgICAgICAgZGV2aWNlcy5hcHBlbmQoCiAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgImluZGV4IjogaW5kZXgsCiAgICAgICAgICAgICAgICAgICAgIm5hbWUiOiBwcm9wcy5uYW1lLAogICAgICAgICAgICAgICAgICAgICJ0b3RhbF9tZW1vcnlfYnl0ZXMiOiBpbnQocHJvcHMudG90YWxfbWVtb3J5KSwKICAgICAgICAgICAgICAgICAgICAiY29tcHV0ZV9jYXBhYmlsaXR5IjogZiJ7cHJvcHMubWFqb3J9Lntwcm9wcy5taW5vcn0iLAogICAgICAgICAgICAgICAgfQogICAgICAgICAgICApCiAgICAgICAgcmVzdWx0WyJncHVfZGV2aWNlcyJdID0gZGV2aWNlcwogICAgICAgIHRvcmNoLmN1ZGEucmVzZXRfcGVha19tZW1vcnlfc3RhdHMoKQogICAgcmV0dXJuIHJlc3VsdAoKCmRlZiBnaXRfaW5mbyhyZXBvX3Jvb3Q6IHN0ciB8IFBhdGgpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgcmVwb19yb290ID0gUGF0aChyZXBvX3Jvb3QpCiAgICB0cnk6CiAgICAgICAgY29tbWl0ID0gc3VicHJvY2Vzcy5ydW4oCiAgICAgICAgICAgIFsiZ2l0IiwgIi1DIiwgc3RyKHJlcG9fcm9vdCksICJyZXYtcGFyc2UiLCAiSEVBRCJdLAogICAgICAgICAgICBjaGVjaz1UcnVlLAogICAgICAgICAgICBjYXB0dXJlX291dHB1dD1UcnVlLAogICAgICAgICAgICB0ZXh0PVRydWUsCiAgICAgICAgKS5zdGRvdXQuc3RyaXAoKQogICAgICAgIHN0YXR1cyA9IHN1YnByb2Nlc3MucnVuKAogICAgICAgICAgICBbImdpdCIsICItQyIsIHN0cihyZXBvX3Jvb3QpLCAic3RhdHVzIiwgIi0tcG9yY2VsYWluIl0sCiAgICAgICAgICAgIGNoZWNrPVRydWUsCiAgICAgICAgICAgIGNhcHR1cmVfb3V0cHV0PVRydWUsCiAgICAgICAgICAgIHRleHQ9VHJ1ZSwKICAgICAgICApLnN0ZG91dAogICAgICAgIHJldHVybiB7ImNvbW1pdCI6IGNvbW1pdCwgIndvcmt0cmVlX2RpcnR5IjogYm9vbChzdGF0dXMuc3RyaXAoKSl9CiAgICBleGNlcHQgKE9TRXJyb3IsIHN1YnByb2Nlc3MuQ2FsbGVkUHJvY2Vzc0Vycm9yKToKICAgICAgICByZXR1cm4geyJjb21taXQiOiBOb25lLCAid29ya3RyZWVfZGlydHkiOiBOb25lfQo=","src/filament_baseline/split.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGNzdgppbXBvcnQgaGFzaGxpYgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpmcm9tIHNrbGVhcm4ubW9kZWxfc2VsZWN0aW9uIGltcG9ydCBTdHJhdGlmaWVkU2h1ZmZsZVNwbGl0Cgpmcm9tIC5kYXRhIGltcG9ydCBDb21wZXRpdGlvbkRhdGFzZXQKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBPYnNlcnZhdGlvblNwbGl0OgogICAgYXNzaWdubWVudHM6IGRpY3Rbc3RyLCBzdHJdCiAgICBtYW5pZmVzdF9wYXRoOiBQYXRoCiAgICBzaGEyNTY6IHN0cgoKICAgIEBwcm9wZXJ0eQogICAgZGVmIHZhbGlkYXRpb25fbmFtZXMoc2VsZikgLT4gdHVwbGVbc3RyLCAuLi5dOgogICAgICAgIHJldHVybiB0dXBsZSgKICAgICAgICAgICAgc29ydGVkKG5hbWUgZm9yIG5hbWUsIHBhcnRpdGlvbiBpbiBzZWxmLmFzc2lnbm1lbnRzLml0ZW1zKCkgaWYgcGFydGl0aW9uID09ICJ2YWxpZGF0aW9uIikKICAgICAgICApCgogICAgQHByb3BlcnR5CiAgICBkZWYgdHJhaW5pbmdfbmFtZXMoc2VsZikgLT4gdHVwbGVbc3RyLCAuLi5dOgogICAgICAgIHJldHVybiB0dXBsZSgKICAgICAgICAgICAgc29ydGVkKG5hbWUgZm9yIG5hbWUsIHBhcnRpdGlvbiBpbiBzZWxmLmFzc2lnbm1lbnRzLml0ZW1zKCkgaWYgcGFydGl0aW9uID09ICJ0cmFpbiIpCiAgICAgICAgKQoKCmRlZiBjcmVhdGVfb2JzZXJ2YXRpb25fc3BsaXQoCiAgICBkYXRhc2V0OiBDb21wZXRpdGlvbkRhdGFzZXQsCiAgICBzcGxpdF9kaXI6IHN0ciB8IFBhdGgsCiAgICAqLAogICAgc2VlZDogaW50ID0gNDIsCiAgICB2YWxpZGF0aW9uX2ZyYWN0aW9uOiBmbG9hdCA9IDAuMiwKKSAtPiBPYnNlcnZhdGlvblNwbGl0OgogICAgbmFtZXMgPSBsaXN0KGRhdGFzZXQub2JzZXJ2YXRpb25fbmFtZXMpCiAgICB5ZWFycyA9IFtkYXRhc2V0LnZpZXdzX2J5X29ic2VydmF0aW9uW25hbWVdWzBdLnllYXIgZm9yIG5hbWUgaW4gbmFtZXNdCiAgICBzcGxpdHRlciA9IFN0cmF0aWZpZWRTaHVmZmxlU3BsaXQoCiAgICAgICAgbl9zcGxpdHM9MSwgdGVzdF9zaXplPXZhbGlkYXRpb25fZnJhY3Rpb24sIHJhbmRvbV9zdGF0ZT1zZWVkCiAgICApCiAgICB0cmFpbl9pbmRpY2VzLCB2YWxpZGF0aW9uX2luZGljZXMgPSBuZXh0KHNwbGl0dGVyLnNwbGl0KG5hbWVzLCB5ZWFycykpCiAgICBhc3NpZ25tZW50cyA9IHtuYW1lOiAidHJhaW4iIGZvciBuYW1lIGluIG5hbWVzfQogICAgZm9yIGluZGV4IGluIHZhbGlkYXRpb25faW5kaWNlczoKICAgICAgICBhc3NpZ25tZW50c1tuYW1lc1tpbnQoaW5kZXgpXV0gPSAidmFsaWRhdGlvbiIKCiAgICByb3dzID0gW10KICAgIGZvciBuYW1lIGluIG5hbWVzOgogICAgICAgIHZpZXcgPSBkYXRhc2V0LnZpZXdzX2J5X29ic2VydmF0aW9uW25hbWVdWzBdCiAgICAgICAgcm93cy5hcHBlbmQoCiAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICJmaWxlX25hbWUiOiBuYW1lLAogICAgICAgICAgICAgICAgInllYXIiOiB2aWV3LnllYXIsCiAgICAgICAgICAgICAgICAic2l0ZSI6IHZpZXcuc2l0ZSwKICAgICAgICAgICAgICAgICJwYXJ0aXRpb24iOiBhc3NpZ25tZW50c1tuYW1lXSwKICAgICAgICAgICAgfQogICAgICAgICkKCiAgICBzcGxpdF9kaXIgPSBQYXRoKHNwbGl0X2RpcikKICAgIHNwbGl0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBtYW5pZmVzdF9wYXRoID0gc3BsaXRfZGlyIC8gZiJvYnNlcnZhdGlvbl9zcGxpdF9zZWVke3NlZWR9LmNzdiIKICAgIGZyb20gaW8gaW1wb3J0IFN0cmluZ0lPCgogICAgYnVmZmVyID0gU3RyaW5nSU8obmV3bGluZT0iIikKICAgIHdyaXRlciA9IGNzdi5EaWN0V3JpdGVyKAogICAgICAgIGJ1ZmZlciwKICAgICAgICBmaWVsZG5hbWVzPSgiZmlsZV9uYW1lIiwgInllYXIiLCAic2l0ZSIsICJwYXJ0aXRpb24iKSwKICAgICAgICBsaW5ldGVybWluYXRvcj0iXG4iLAogICAgKQogICAgd3JpdGVyLndyaXRlaGVhZGVyKCkKICAgIHdyaXRlci53cml0ZXJvd3Mocm93cykKICAgIG1hbmlmZXN0X2J5dGVzID0gYnVmZmVyLmdldHZhbHVlKCkuZW5jb2RlKCJ1dGYtOCIpCiAgICBkaWdlc3QgPSBoYXNobGliLnNoYTI1NihtYW5pZmVzdF9ieXRlcykuaGV4ZGlnZXN0KCkKCiAgICBpZiBtYW5pZmVzdF9wYXRoLmV4aXN0cygpOgogICAgICAgIGV4aXN0aW5nID0gbWFuaWZlc3RfcGF0aC5yZWFkX2J5dGVzKCkKICAgICAgICBpZiBleGlzdGluZyAhPSBtYW5pZmVzdF9ieXRlczoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKAogICAgICAgICAgICAgICAgZiJFeGlzdGluZyBzcGxpdCBtYW5pZmVzdCBkaWZmZXJzIGZyb20gdGhlIGRldGVybWluaXN0aWMgc3BsaXQ6IHttYW5pZmVzdF9wYXRofSIKICAgICAgICAgICAgKQogICAgZWxzZToKICAgICAgICBtYW5pZmVzdF9wYXRoLndyaXRlX2J5dGVzKG1hbmlmZXN0X2J5dGVzKQoKICAgIGlmIHNldChhc3NpZ25tZW50cy52YWx1ZXMoKSkgIT0geyJ0cmFpbiIsICJ2YWxpZGF0aW9uIn06CiAgICAgICAgcmFpc2UgQXNzZXJ0aW9uRXJyb3IoIlRoZSBzcGxpdCBtdXN0IGNvbnRhaW4gYm90aCB0cmFpbmluZyBhbmQgdmFsaWRhdGlvbiBvYnNlcnZhdGlvbnMiKQogICAgaWYgc2V0KGFzc2lnbm1lbnRzKSAhPSBzZXQobmFtZXMpOgogICAgICAgIHJhaXNlIEFzc2VydGlvbkVycm9yKCJUaGUgc3BsaXQgbWFuaWZlc3QgZG9lcyBub3QgY292ZXIgZXZlcnkgb3JpZ2luYWwgb2JzZXJ2YXRpb24iKQogICAgaWYgc2V0KGRhdGFzZXQub2JzZXJ2YXRpb25fbmFtZXMpICE9IHNldChhc3NpZ25tZW50cyk6CiAgICAgICAgcmFpc2UgQXNzZXJ0aW9uRXJyb3IoIk9ic2VydmF0aW9uIHNwbGl0IGtleXMgZG8gbm90IG1hdGNoIENPQ08gZmlsZV9uYW1lIGdyb3VwcyIpCgogICAgcmV0dXJuIE9ic2VydmF0aW9uU3BsaXQoYXNzaWdubWVudHMsIG1hbmlmZXN0X3BhdGgsIGRpZ2VzdCkKCgpkZWYgbG9hZF9vYnNlcnZhdGlvbl9zcGxpdF9tYW5pZmVzdCgKICAgIGRhdGFzZXQ6IENvbXBldGl0aW9uRGF0YXNldCwKICAgIG1hbmlmZXN0X3BhdGg6IHN0ciB8IFBhdGgsCiAgICAqLAogICAgYWxsb3dfc3Vic2V0OiBib29sID0gRmFsc2UsCikgLT4gT2JzZXJ2YXRpb25TcGxpdDoKICAgICIiIkxvYWQgYW4gZXhwbGljaXRseSBncm91cGVkIHRyYWluL3ZhbGlkYXRpb24gbWFuaWZlc3QgdW5kZXIgZGF0YS9zcGxpdHMuIiIiCiAgICBtYW5pZmVzdF9wYXRoID0gUGF0aChtYW5pZmVzdF9wYXRoKS5leHBhbmR1c2VyKCkucmVzb2x2ZSgpCiAgICBjb250ZW50ID0gbWFuaWZlc3RfcGF0aC5yZWFkX2J5dGVzKCkKICAgIGFzc2lnbm1lbnRzOiBkaWN0W3N0ciwgc3RyXSA9IHt9CiAgICB3aXRoIG1hbmlmZXN0X3BhdGgub3BlbigiciIsIGVuY29kaW5nPSJ1dGYtOCIsIG5ld2xpbmU9IiIpIGFzIHN0cmVhbToKICAgICAgICByZWFkZXIgPSBjc3YuRGljdFJlYWRlcihzdHJlYW0pCiAgICAgICAgaWYgcmVhZGVyLmZpZWxkbmFtZXMgIT0gWyJmaWxlX25hbWUiLCAieWVhciIsICJzaXRlIiwgInBhcnRpdGlvbiJdOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJTcGxpdCBtYW5pZmVzdCBtdXN0IGNvbnRhaW4gZmlsZV9uYW1lLCB5ZWFyLCBzaXRlLCBwYXJ0aXRpb24gY29sdW1ucyIpCiAgICAgICAgZm9yIHJvdyBpbiByZWFkZXI6CiAgICAgICAgICAgIG5hbWUgPSByb3dbImZpbGVfbmFtZSJdCiAgICAgICAgICAgIGlmIG5hbWUgaW4gYXNzaWdubWVudHM6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiRHVwbGljYXRlIG9ic2VydmF0aW9uIGluIHNwbGl0IG1hbmlmZXN0OiB7bmFtZX0iKQogICAgICAgICAgICBpZiBuYW1lIG5vdCBpbiBkYXRhc2V0LnZpZXdzX2J5X29ic2VydmF0aW9uOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlVua25vd24gb2JzZXJ2YXRpb24gaW4gc3BsaXQgbWFuaWZlc3Q6IHtuYW1lfSIpCiAgICAgICAgICAgIG9yaWdpbmFsX3ZpZXcgPSBkYXRhc2V0LnZpZXdzX2J5X29ic2VydmF0aW9uW25hbWVdWzBdCiAgICAgICAgICAgIGlmIGludChyb3dbInllYXIiXSkgIT0gb3JpZ2luYWxfdmlldy55ZWFyIG9yIHJvd1sic2l0ZSJdICE9IG9yaWdpbmFsX3ZpZXcuc2l0ZToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJZZWFyL3NpdGUgbWV0YWRhdGEgY2hhbmdlZCBmb3Igb2JzZXJ2YXRpb24ge25hbWV9IikKICAgICAgICAgICAgaWYgcm93WyJwYXJ0aXRpb24iXSBub3QgaW4geyJ0cmFpbiIsICJ2YWxpZGF0aW9uIn06CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiSW52YWxpZCBwYXJ0aXRpb24gZm9yIG9ic2VydmF0aW9uIHtuYW1lfToge3Jvd1sncGFydGl0aW9uJ119IikKICAgICAgICAgICAgYXNzaWdubWVudHNbbmFtZV0gPSByb3dbInBhcnRpdGlvbiJdCgogICAgZGF0YXNldF9uYW1lcyA9IHNldChkYXRhc2V0Lm9ic2VydmF0aW9uX25hbWVzKQogICAgbWFuaWZlc3RfbmFtZXMgPSBzZXQoYXNzaWdubWVudHMpCiAgICBpZiBub3QgbWFuaWZlc3RfbmFtZXM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiU3BsaXQgbWFuaWZlc3QgY29udGFpbnMgbm8gb2JzZXJ2YXRpb25zIikKICAgIGlmIGFsbG93X3N1YnNldDoKICAgICAgICBpZiBub3QgbWFuaWZlc3RfbmFtZXMuaXNzdWJzZXQoZGF0YXNldF9uYW1lcyk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlNwbGl0IG1hbmlmZXN0IGNvbnRhaW5zIG9ic2VydmF0aW9ucyBvdXRzaWRlIHRoZSBjb21wZXRpdGlvbiBkYXRhc2V0IikKICAgIGVsaWYgbWFuaWZlc3RfbmFtZXMgIT0gZGF0YXNldF9uYW1lczoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJTcGxpdCBtYW5pZmVzdCBtdXN0IGNvdmVyIGV2ZXJ5IGNvbXBldGl0aW9uIHRyYWluaW5nIG9ic2VydmF0aW9uIikKICAgIGlmIHNldChhc3NpZ25tZW50cy52YWx1ZXMoKSkgIT0geyJ0cmFpbiIsICJ2YWxpZGF0aW9uIn06CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiU3BsaXQgbWFuaWZlc3QgbXVzdCBjb250YWluIGJvdGggdHJhaW4gYW5kIHZhbGlkYXRpb24gb2JzZXJ2YXRpb25zIikKCiAgICByZXR1cm4gT2JzZXJ2YXRpb25TcGxpdCgKICAgICAgICBhc3NpZ25tZW50cz1hc3NpZ25tZW50cywKICAgICAgICBtYW5pZmVzdF9wYXRoPW1hbmlmZXN0X3BhdGgsCiAgICAgICAgc2hhMjU2PWhhc2hsaWIuc2hhMjU2KGNvbnRlbnQpLmhleGRpZ2VzdCgpLAogICAgKQo=","scripts/export_yolo_dataset.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgc2h1dGlsCmltcG9ydCBzeXMKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKClJFUE9fUk9PVCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzFdCnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUkVQT19ST09UIC8gInNyYyIpKQoKZnJvbSBmaWxhbWVudF9iYXNlbGluZS5kYXRhIGltcG9ydCBDb21wZXRpdGlvbkRhdGFzZXQsIGxvYWRfZGF0YXNldApmcm9tIGZpbGFtZW50X2Jhc2VsaW5lLnNwbGl0IGltcG9ydCBPYnNlcnZhdGlvblNwbGl0LCBjcmVhdGVfb2JzZXJ2YXRpb25fc3BsaXQKCgpkZWYgX3NhZmVfdmlld19uYW1lKGZpbGVfbmFtZTogc3RyLCBpbWFnZV9pZDogc3RyKSAtPiBzdHI6CiAgICB2aWV3X2RpZ2VzdCA9IGhhc2hsaWIuc2hhMShpbWFnZV9pZC5lbmNvZGUoInV0Zi04IikpLmhleGRpZ2VzdCgpWzoxMl0KICAgIHJldHVybiBmIntQYXRoKGZpbGVfbmFtZSkuc3RlbX1fX3t2aWV3X2RpZ2VzdH0iCgoKZGVmIF9wb2x5Z29uX2FyZWEocG9seWdvbjogbGlzdFtmbG9hdF0pIC0+IGZsb2F0OgogICAgcG9pbnRzID0gbGlzdCh6aXAocG9seWdvblswOjoyXSwgcG9seWdvblsxOjoyXSwgc3RyaWN0PVRydWUpKQogICAgcmV0dXJuIGFicygKICAgICAgICBzdW0oCiAgICAgICAgICAgIHBvaW50c1tpbmRleF1bMF0gKiBwb2ludHNbKGluZGV4ICsgMSkgJSBsZW4ocG9pbnRzKV1bMV0KICAgICAgICAgICAgLSBwb2ludHNbaW5kZXhdWzFdICogcG9pbnRzWyhpbmRleCArIDEpICUgbGVuKHBvaW50cyldWzBdCiAgICAgICAgICAgIGZvciBpbmRleCBpbiByYW5nZShsZW4ocG9pbnRzKSkKICAgICAgICApCiAgICApICogMC41CgoKZGVmIF9sYWJlbHNfZm9yX3ZpZXcoZGF0YXNldDogQ29tcGV0aXRpb25EYXRhc2V0LCBpbWFnZV9pZDogc3RyKSAtPiBsaXN0W3N0cl06CiAgICB2aWV3ID0gbmV4dCh2aWV3IGZvciB2aWV3IGluIGRhdGFzZXQuaW1hZ2Vfdmlld3MgaWYgdmlldy5pbWFnZV9pZCA9PSBpbWFnZV9pZCkKICAgIG91dHB1dCA9IFtdCiAgICBmb3IgYW5ub3RhdGlvbiBpbiBkYXRhc2V0LmFubm90YXRpb25zX2J5X2ltYWdlX2lkW2ltYWdlX2lkXToKICAgICAgICBzZWdtZW50YXRpb24gPSBhbm5vdGF0aW9uLmdldCgic2VnbWVudGF0aW9uIikKICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShzZWdtZW50YXRpb24sIGxpc3QpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiRXhwZWN0ZWQgcG9seWdvbiBhbm5vdGF0aW9ucyBmb3IgWU9MTyBleHBvcnQ6IHt2aWV3LmZpbGVfbmFtZX0iKQogICAgICAgIHBvbHlnb25zID0gWwogICAgICAgICAgICBwb2x5Z29uCiAgICAgICAgICAgIGZvciBwb2x5Z29uIGluIHNlZ21lbnRhdGlvbgogICAgICAgICAgICBpZiBpc2luc3RhbmNlKHBvbHlnb24sIGxpc3QpIGFuZCBsZW4ocG9seWdvbikgPj0gNiBhbmQgbGVuKHBvbHlnb24pICUgMiA9PSAwCiAgICAgICAgXQogICAgICAgIGlmIG5vdCBwb2x5Z29uczoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBwb2x5Z29uID0gbWF4KHBvbHlnb25zLCBrZXk9X3BvbHlnb25fYXJlYSkKICAgICAgICB2YWx1ZXMgPSBbXQogICAgICAgIGZvciB4LCB5IGluIHppcChwb2x5Z29uWzA6OjJdLCBwb2x5Z29uWzE6OjJdLCBzdHJpY3Q9VHJ1ZSk6CiAgICAgICAgICAgIHZhbHVlcy5leHRlbmQoCiAgICAgICAgICAgICAgICAoCiAgICAgICAgICAgICAgICAgICAgbWluKG1heChmbG9hdCh4KSAvIHZpZXcud2lkdGgsIDAuMCksIDEuMCksCiAgICAgICAgICAgICAgICAgICAgbWluKG1heChmbG9hdCh5KSAvIHZpZXcuaGVpZ2h0LCAwLjApLCAxLjApLAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICApCiAgICAgICAgb3V0cHV0LmFwcGVuZCgiMCAiICsgIiAiLmpvaW4oZiJ7dmFsdWU6LjdmfSIgZm9yIHZhbHVlIGluIHZhbHVlcykpCiAgICByZXR1cm4gb3V0cHV0CgoKZGVmIF9saW5rX29yX2NvcHkoc291cmNlOiBQYXRoLCBkZXN0aW5hdGlvbjogUGF0aCkgLT4gTm9uZToKICAgIGRlc3RpbmF0aW9uLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBpZiBkZXN0aW5hdGlvbi5leGlzdHMoKToKICAgICAgICByZXR1cm4KICAgIHRyeToKICAgICAgICBvcy5saW5rKHNvdXJjZSwgZGVzdGluYXRpb24pCiAgICBleGNlcHQgT1NFcnJvcjoKICAgICAgICBzaHV0aWwuY29weTIoc291cmNlLCBkZXN0aW5hdGlvbikKCgpkZWYgX3dpdGhpbihwYXRoOiBQYXRoLCByb290OiBQYXRoKSAtPiBib29sOgogICAgdHJ5OgogICAgICAgIHBhdGgucmVzb2x2ZSgpLnJlbGF0aXZlX3RvKHJvb3QucmVzb2x2ZSgpKQogICAgICAgIHJldHVybiBUcnVlCiAgICBleGNlcHQgVmFsdWVFcnJvcjoKICAgICAgICByZXR1cm4gRmFsc2UKCgpkZWYgZXhwb3J0X3lvbG9fZGF0YXNldCgKICAgIGRhdGFzZXQ6IENvbXBldGl0aW9uRGF0YXNldCwKICAgIHNwbGl0OiBPYnNlcnZhdGlvblNwbGl0LAogICAgb3V0cHV0X3Jvb3Q6IHN0ciB8IFBhdGgsCiAgICAqLAogICAgbGltaXRfcGVyX3BhcnRpdGlvbjogaW50IHwgTm9uZSA9IE5vbmUsCikgLT4gZGljdFtzdHIsIEFueV06CiAgICBvdXRwdXRfcm9vdCA9IFBhdGgob3V0cHV0X3Jvb3QpLmV4cGFuZHVzZXIoKS5yZXNvbHZlKCkKICAgIHByb2Nlc3NlZF9yb290ID0gKFJFUE9fUk9PVCAvICJkYXRhIiAvICJwcm9jZXNzZWQiKS5yZXNvbHZlKCkKICAgIGlmIG5vdCBfd2l0aGluKG91dHB1dF9yb290LCBwcm9jZXNzZWRfcm9vdCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkV4cG9ydGVkIGRldGVjdG9yIGRhdGEgbXVzdCByZW1haW4gdW5kZXIgZGF0YS9wcm9jZXNzZWQ6IHtvdXRwdXRfcm9vdH0iKQogICAgaWYgbGltaXRfcGVyX3BhcnRpdGlvbiBpcyBub3QgTm9uZSBhbmQgbGltaXRfcGVyX3BhcnRpdGlvbiA8IDE6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibGltaXRfcGVyX3BhcnRpdGlvbiBtdXN0IGJlIHBvc2l0aXZlIikKICAgIHJvd3MgPSBbXQogICAgY291bnRzID0geyJ0cmFpbiI6IHsidmlld3MiOiAwLCAiaW5zdGFuY2VzIjogMH0sICJ2YWwiOiB7InZpZXdzIjogMCwgImluc3RhbmNlcyI6IDB9fQogICAgcGFydGl0aW9ucyA9IHsidHJhaW4iOiBzZXQoc3BsaXQudHJhaW5pbmdfbmFtZXMpLCAidmFsIjogc2V0KHNwbGl0LnZhbGlkYXRpb25fbmFtZXMpfQogICAgZm9yIHBhcnRpdGlvbiwgb2JzZXJ2YXRpb25fbmFtZXMgaW4gcGFydGl0aW9ucy5pdGVtcygpOgogICAgICAgIHZpZXdzID0gW3ZpZXcgZm9yIHZpZXcgaW4gZGF0YXNldC5pbWFnZV92aWV3cyBpZiB2aWV3LmZpbGVfbmFtZSBpbiBvYnNlcnZhdGlvbl9uYW1lc10KICAgICAgICB2aWV3cy5zb3J0KGtleT1sYW1iZGEgdmlldzogKHZpZXcuZmlsZV9uYW1lLCB2aWV3LmltYWdlX2lkKSkKICAgICAgICBpZiBsaW1pdF9wZXJfcGFydGl0aW9uIGlzIG5vdCBOb25lOgogICAgICAgICAgICB2aWV3cyA9IHZpZXdzWzpsaW1pdF9wZXJfcGFydGl0aW9uXQogICAgICAgIGZvciB2aWV3IGluIHZpZXdzOgogICAgICAgICAgICBzYW1wbGUgPSBfc2FmZV92aWV3X25hbWUodmlldy5maWxlX25hbWUsIHZpZXcuaW1hZ2VfaWQpCiAgICAgICAgICAgIGltYWdlX3NvdXJjZSA9IGRhdGFzZXQudHJhaW5faW1hZ2VfZGlyIC8gdmlldy5maWxlX25hbWUKICAgICAgICAgICAgaW1hZ2VfdGFyZ2V0ID0gb3V0cHV0X3Jvb3QgLyAiaW1hZ2VzIiAvIHBhcnRpdGlvbiAvIGYie3NhbXBsZX0uanBnIgogICAgICAgICAgICBsYWJlbF90YXJnZXQgPSBvdXRwdXRfcm9vdCAvICJsYWJlbHMiIC8gcGFydGl0aW9uIC8gZiJ7c2FtcGxlfS50eHQiCiAgICAgICAgICAgIF9saW5rX29yX2NvcHkoaW1hZ2Vfc291cmNlLCBpbWFnZV90YXJnZXQpCiAgICAgICAgICAgIGxhYmVscyA9IF9sYWJlbHNfZm9yX3ZpZXcoZGF0YXNldCwgdmlldy5pbWFnZV9pZCkKICAgICAgICAgICAgbGFiZWxfdGFyZ2V0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgICAgIGxhYmVsX3RhcmdldC53cml0ZV90ZXh0KCJcbiIuam9pbihsYWJlbHMpICsgIlxuIiwgZW5jb2Rpbmc9InV0Zi04IikKICAgICAgICAgICAgY291bnRzW3BhcnRpdGlvbl1bInZpZXdzIl0gKz0gMQogICAgICAgICAgICBjb3VudHNbcGFydGl0aW9uXVsiaW5zdGFuY2VzIl0gKz0gbGVuKGxhYmVscykKICAgICAgICAgICAgcm93cy5hcHBlbmQoCiAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgInBhcnRpdGlvbiI6IHBhcnRpdGlvbiwKICAgICAgICAgICAgICAgICAgICAib2JzZXJ2YXRpb24iOiB2aWV3LmZpbGVfbmFtZSwKICAgICAgICAgICAgICAgICAgICAidmlld19pZCI6IHZpZXcuaW1hZ2VfaWQsCiAgICAgICAgICAgICAgICAgICAgInNhbXBsZSI6IHNhbXBsZSwKICAgICAgICAgICAgICAgICAgICAiaW5zdGFuY2VzIjogbGVuKGxhYmVscyksCiAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICkKCiAgICBvdXRwdXRfcm9vdC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBkYXRhX3lhbWwgPSAoCiAgICAgICAgZiJwYXRoOiB7b3V0cHV0X3Jvb3QuYXNfcG9zaXgoKX1cbiIKICAgICAgICAidHJhaW46IGltYWdlcy90cmFpblxuIgogICAgICAgICJ2YWw6IGltYWdlcy92YWxcbiIKICAgICAgICAibmFtZXM6XG4gIDA6IGZpbGFtZW50XG4iCiAgICApCiAgICAob3V0cHV0X3Jvb3QgLyAiZGF0YS55YW1sIikud3JpdGVfdGV4dChkYXRhX3lhbWwsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICB2aWV3X21hbmlmZXN0ID0ganNvbi5kdW1wcyhyb3dzLCBzb3J0X2tleXM9VHJ1ZSwgc2VwYXJhdG9ycz0oIiwiLCAiOiIpKS5lbmNvZGUoInV0Zi04IikKICAgIHJlc3VsdCA9IHsKICAgICAgICAic3BsaXRfc2hhMjU2Ijogc3BsaXQuc2hhMjU2LAogICAgICAgICJsaW1pdF9wZXJfcGFydGl0aW9uIjogbGltaXRfcGVyX3BhcnRpdGlvbiwKICAgICAgICAiZ3JvdXBpbmciOiAib3JpZ2luYWwgb2JzZXJ2YXRpb247IGFsbCBhbm5vdGF0b3Igdmlld3Mgc3RheSB0b2dldGhlciIsCiAgICAgICAgInRyYWluX29ic2VydmF0aW9ucyI6IGxlbihzcGxpdC50cmFpbmluZ19uYW1lcyksCiAgICAgICAgInZhbGlkYXRpb25fb2JzZXJ2YXRpb25zIjogbGVuKHNwbGl0LnZhbGlkYXRpb25fbmFtZXMpLAogICAgICAgICJjb3VudHMiOiBjb3VudHMsCiAgICAgICAgInZpZXdfbWFuaWZlc3Rfc2hhMjU2IjogaGFzaGxpYi5zaGEyNTYodmlld19tYW5pZmVzdCkuaGV4ZGlnZXN0KCksCiAgICAgICAgImRhdGFfeWFtbCI6IHN0cihvdXRwdXRfcm9vdCAvICJkYXRhLnlhbWwiKSwKICAgICAgICAidmlld3MiOiByb3dzLAogICAgfQogICAgKG91dHB1dF9yb290IC8gImV4cG9ydF9tYW5pZmVzdC5qc29uIikud3JpdGVfdGV4dCgKICAgICAgICBqc29uLmR1bXBzKHJlc3VsdCwgaW5kZW50PTIsIHNvcnRfa2V5cz1UcnVlKSArICJcbiIsIGVuY29kaW5nPSJ1dGYtOCIKICAgICkKICAgIHJldHVybiByZXN1bHQKCgpkZWYgbWFpbigpIC0+IGludDoKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJFeHBvcnQgZ3JvdXBlZCBDT0NPIGluc3RhbmNlIGFubm90YXRpb25zIHRvIFlPTE8gc2VnbWVudGF0aW9uIGZvcm1hdC4iKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kYXRhLXJvb3QiLCB0eXBlPVBhdGgsIGRlZmF1bHQ9Tm9uZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3V0cHV0LXJvb3QiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXNwbGl0LWRpciIsIHR5cGU9UGF0aCwgZGVmYXVsdD1SRVBPX1JPT1QgLyAiZGF0YSIgLyAic3BsaXRzIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbGltaXQtcGVyLXBhcnRpdGlvbiIsIHR5cGU9aW50LCBkZWZhdWx0PU5vbmUpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgZGF0YV9yb290ID0gKGFyZ3MuZGF0YV9yb290IG9yIFJFUE9fUk9PVCAvICJkYXRhIiAvICJyYXciIC8gIk1BR0ZpTE9fMS4wX0thZ2dsZV8yMDI2IikuZXhwYW5kdXNlcigpLnJlc29sdmUoKQogICAgb3V0cHV0X3Jvb3QgPSBhcmdzLm91dHB1dF9yb290LmV4cGFuZHVzZXIoKS5yZXNvbHZlKCkKICAgIHNwbGl0X2RpciA9IGFyZ3Muc3BsaXRfZGlyLmV4cGFuZHVzZXIoKS5yZXNvbHZlKCkKICAgIGlmIG5vdCBkYXRhX3Jvb3QuaXNfcmVsYXRpdmVfdG8oKFJFUE9fUk9PVCAvICJkYXRhIikucmVzb2x2ZSgpKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiTG9jYWwgY29tcGV0aXRpb24gZGF0YSBtdXN0IHJlbWFpbiB1bmRlciBkYXRhLzoge2RhdGFfcm9vdH0iKQogICAgaWYgbm90IG91dHB1dF9yb290LmlzX3JlbGF0aXZlX3RvKChSRVBPX1JPT1QgLyAiZGF0YSIgLyAicHJvY2Vzc2VkIikucmVzb2x2ZSgpKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiTG9jYWwgZXhwb3J0ZWQgZGF0YSBtdXN0IHJlbWFpbiB1bmRlciBkYXRhL3Byb2Nlc3NlZC86IHtvdXRwdXRfcm9vdH0iKQogICAgaWYgbm90IHNwbGl0X2Rpci5pc19yZWxhdGl2ZV90bygoUkVQT19ST09UIC8gImRhdGEiIC8gInNwbGl0cyIpLnJlc29sdmUoKSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkxvY2FsIHNwbGl0IGZpbGVzIG11c3QgcmVtYWluIHVuZGVyIGRhdGEvc3BsaXRzLzoge3NwbGl0X2Rpcn0iKQogICAgZGF0YXNldCA9IGxvYWRfZGF0YXNldChkYXRhX3Jvb3QpCiAgICBjb25maWcgPSBqc29uLmxvYWRzKChSRVBPX1JPT1QgLyAiY29uZmlncyIgLyAidmFsaWRhdGlvbi1iYXNlbGluZS5qc29uIikucmVhZF90ZXh0KCkpCiAgICBzcGxpdCA9IGNyZWF0ZV9vYnNlcnZhdGlvbl9zcGxpdCgKICAgICAgICBkYXRhc2V0LAogICAgICAgIHNwbGl0X2RpciwKICAgICAgICBzZWVkPWludChjb25maWdbInNlZWQiXSksCiAgICAgICAgdmFsaWRhdGlvbl9mcmFjdGlvbj1mbG9hdChjb25maWdbInZhbGlkYXRpb25fZnJhY3Rpb24iXSksCiAgICApCiAgICBpZiBzcGxpdC5zaGEyNTYgIT0gY29uZmlnWyJleHBlY3RlZF9zcGxpdF9zaGEyNTYiXToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJGcm96ZW4gc3BsaXQgY2hhbmdlZDoge3NwbGl0LnNoYTI1Nn0iKQogICAgcmVzdWx0ID0gZXhwb3J0X3lvbG9fZGF0YXNldCgKICAgICAgICBkYXRhc2V0LAogICAgICAgIHNwbGl0LAogICAgICAgIG91dHB1dF9yb290LAogICAgICAgIGxpbWl0X3Blcl9wYXJ0aXRpb249YXJncy5saW1pdF9wZXJfcGFydGl0aW9uLAogICAgKQogICAgcHJpbnQoanNvbi5kdW1wcyh7a2V5OiB2YWx1ZSBmb3Iga2V5LCB2YWx1ZSBpbiByZXN1bHQuaXRlbXMoKSBpZiBrZXkgIT0gInZpZXdzIn0sIGluZGVudD0yLCBzb3J0X2tleXM9VHJ1ZSkpCiAgICByZXR1cm4gMAoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICByYWlzZSBTeXN0ZW1FeGl0KG1haW4oKSkK","scripts/run_yolo_swefil_pretrain.py":"ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBjc3YKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCByYW5kb20KaW1wb3J0IHN5cwppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKaW1wb3J0IG51bXB5IGFzIG5wCgpSRVBPX1JPT1QgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1sxXQpzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFJFUE9fUk9PVCAvICJzcmMiKSkKc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihSRVBPX1JPT1QgLyAic2NyaXB0cyIpKQoKZnJvbSBmaWxhbWVudF9iYXNlbGluZS5kYXRhIGltcG9ydCBsb2FkX2RhdGFzZXQKZnJvbSBmaWxhbWVudF9iYXNlbGluZS5leHRlcm5hbF9kYXRhIGltcG9ydCBsb2FkX3N3ZWZpbF9kYXRhc2V0CmZyb20gZmlsYW1lbnRfYmFzZWxpbmUucnVudGltZSBpbXBvcnQgaGFyZHdhcmVfaW5mbywgcGFja2FnZV92ZXJzaW9ucwpmcm9tIGZpbGFtZW50X2Jhc2VsaW5lLnNwbGl0IGltcG9ydCBPYnNlcnZhdGlvblNwbGl0LCBjcmVhdGVfb2JzZXJ2YXRpb25fc3BsaXQKZnJvbSBleHBvcnRfeW9sb19kYXRhc2V0IGltcG9ydCBleHBvcnRfeW9sb19kYXRhc2V0CgoKZGVmIF9zaGEyNTYocGF0aDogUGF0aCkgLT4gc3RyOgogICAgZGlnZXN0ID0gaGFzaGxpYi5zaGEyNTYoKQogICAgd2l0aCBwYXRoLm9wZW4oInJiIikgYXMgc3RyZWFtOgogICAgICAgIGZvciBibG9jayBpbiBpdGVyKGxhbWJkYTogc3RyZWFtLnJlYWQoMTAyNCAqIDEwMjQpLCBiIiIpOgogICAgICAgICAgICBkaWdlc3QudXBkYXRlKGJsb2NrKQogICAgcmV0dXJuIGRpZ2VzdC5oZXhkaWdlc3QoKQoKCmRlZiBfZGF0ZV9ncm91cF9zcGxpdCgKICAgIGRhdGFzZXQsCiAgICBtYW5pZmVzdF9wYXRoOiBQYXRoLAogICAgKiwKICAgIHNlZWQ6IGludCwKICAgIHZhbGlkYXRpb25fZnJhY3Rpb246IGZsb2F0LAopIC0+IE9ic2VydmF0aW9uU3BsaXQ6CiAgICBkYXlzID0ge25hbWU6IFBhdGgobmFtZSkuc3RlbVs6OF0gZm9yIG5hbWUgaW4gZGF0YXNldC5vYnNlcnZhdGlvbl9uYW1lc30KICAgIHVuaXF1ZV9kYXlzID0gc29ydGVkKHNldChkYXlzLnZhbHVlcygpKSkKICAgIHZhbGlkYXRpb25fZGF5cyA9IHsKICAgICAgICBkYXkKICAgICAgICBmb3IgZGF5IGluIHVuaXF1ZV9kYXlzCiAgICAgICAgaWYgaW50LmZyb21fYnl0ZXMoCiAgICAgICAgICAgIGhhc2hsaWIuc2hhMjU2KGYic3dlZmlsLXlvbG8tZGF0ZS1zcGxpdC17c2VlZH06e2RheX0iLmVuY29kZSgpKS5kaWdlc3QoKVs6OF0sCiAgICAgICAgICAgICJiaWciLAogICAgICAgICkKICAgICAgICAvIDIqKjY0CiAgICAgICAgPCB2YWxpZGF0aW9uX2ZyYWN0aW9uCiAgICB9CiAgICBpZiBub3QgdmFsaWRhdGlvbl9kYXlzIG9yIHZhbGlkYXRpb25fZGF5cyA9PSBzZXQodW5pcXVlX2RheXMpOgogICAgICAgIHZhbGlkYXRpb25fZGF5cyA9IHNldCh1bmlxdWVfZGF5c1s6OjVdKQogICAgYXNzaWdubWVudHMgPSB7CiAgICAgICAgbmFtZTogKCJ2YWxpZGF0aW9uIiBpZiBkYXlzW25hbWVdIGluIHZhbGlkYXRpb25fZGF5cyBlbHNlICJ0cmFpbiIpCiAgICAgICAgZm9yIG5hbWUgaW4gZGF0YXNldC5vYnNlcnZhdGlvbl9uYW1lcwogICAgfQogICAgaWYgc2V0KGFzc2lnbm1lbnRzLnZhbHVlcygpKSAhPSB7InRyYWluIiwgInZhbGlkYXRpb24ifToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJFeHRlcm5hbCBkYXRlIHNwbGl0IG11c3QgaGF2ZSBub25lbXB0eSB0cmFpbiBhbmQgdmFsaWRhdGlvbiBncm91cHMiKQogICAgdHJhaW5fZGF5cyA9IHtkYXlzW25hbWVdIGZvciBuYW1lLCB2YWx1ZSBpbiBhc3NpZ25tZW50cy5pdGVtcygpIGlmIHZhbHVlID09ICJ0cmFpbiJ9CiAgICB2YWxfZGF5cyA9IHtkYXlzW25hbWVdIGZvciBuYW1lLCB2YWx1ZSBpbiBhc3NpZ25tZW50cy5pdGVtcygpIGlmIHZhbHVlID09ICJ2YWxpZGF0aW9uIn0KICAgIGlmIHRyYWluX2RheXMgJiB2YWxfZGF5czoKICAgICAgICByYWlzZSBBc3NlcnRpb25FcnJvcigiU1dFRmlsIG9ic2VydmF0aW9ucyBmcm9tIG9uZSBjYWxlbmRhciBkYXRlIGNyb3NzZWQgcGFydGl0aW9ucyIpCgogICAgbWFuaWZlc3RfcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgd2l0aCBtYW5pZmVzdF9wYXRoLm9wZW4oInciLCBlbmNvZGluZz0idXRmLTgiLCBuZXdsaW5lPSIiKSBhcyBzdHJlYW06CiAgICAgICAgd3JpdGVyID0gY3N2LkRpY3RXcml0ZXIoCiAgICAgICAgICAgIHN0cmVhbSwKICAgICAgICAgICAgZmllbGRuYW1lcz0oImZpbGVfbmFtZSIsICJ5ZWFyIiwgInNpdGUiLCAicGFydGl0aW9uIiksCiAgICAgICAgICAgIGxpbmV0ZXJtaW5hdG9yPSJcbiIsCiAgICAgICAgKQogICAgICAgIHdyaXRlci53cml0ZWhlYWRlcigpCiAgICAgICAgZm9yIG5hbWUgaW4gc29ydGVkKGFzc2lnbm1lbnRzKToKICAgICAgICAgICAgdmlldyA9IGRhdGFzZXQudmlld3NfYnlfb2JzZXJ2YXRpb25bbmFtZV1bMF0KICAgICAgICAgICAgd3JpdGVyLndyaXRlcm93KAogICAgICAgICAgICAgICAgewogICAgICAgICAgICAgICAgICAgICJmaWxlX25hbWUiOiBuYW1lLAogICAgICAgICAgICAgICAgICAgICJ5ZWFyIjogdmlldy55ZWFyLAogICAgICAgICAgICAgICAgICAgICJzaXRlIjogdmlldy5zaXRlLAogICAgICAgICAgICAgICAgICAgICJwYXJ0aXRpb24iOiBhc3NpZ25tZW50c1tuYW1lXSwKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgKQogICAgcmV0dXJuIE9ic2VydmF0aW9uU3BsaXQoCiAgICAgICAgYXNzaWdubWVudHM9YXNzaWdubWVudHMsCiAgICAgICAgbWFuaWZlc3RfcGF0aD1tYW5pZmVzdF9wYXRoLAogICAgICAgIHNoYTI1Nj1fc2hhMjU2KG1hbmlmZXN0X3BhdGgpLAogICAgKQoKCmRlZiBydW4oYXJnczogYXJncGFyc2UuTmFtZXNwYWNlKSAtPiBkaWN0W3N0ciwgQW55XToKICAgIHJlcG9fcm9vdCA9IGFyZ3MucmVwb19yb290LmV4cGFuZHVzZXIoKS5yZXNvbHZlKCkKICAgIGRhdGFfcm9vdCA9IChhcmdzLmRhdGFfcm9vdCBvciByZXBvX3Jvb3QgLyAiZGF0YSIgLyAicmF3IiAvICJNQUdGaUxPXzEuMF9LYWdnbGVfMjAyNiIpLmV4cGFuZHVzZXIoKS5yZXNvbHZlKCkKICAgIHN3ZWZpbF9yb290ID0gKGFyZ3Muc3dlZmlsX3Jvb3Qgb3IgcmVwb19yb290IC8gImRhdGEiIC8gImV4dGVybmFsIiAvICJTV0VGaWwiKS5leHBhbmR1c2VyKCkucmVzb2x2ZSgpCiAgICBtb2RlbF9wYXRoID0gYXJncy5tb2RlbC5leHBhbmR1c2VyKCkucmVzb2x2ZSgpCiAgICBydW5fcm9vdCA9IChyZXBvX3Jvb3QgLyAiYXJ0aWZhY3RzIiAvICJkZXRlY3Rvci1leHBlcmltZW50cyIgLyBhcmdzLnJ1bl9uYW1lKS5yZXNvbHZlKCkKICAgIGV4cG9ydF9yb290ID0gKHJlcG9fcm9vdCAvICJkYXRhIiAvICJwcm9jZXNzZWQiIC8gInlvbG8iIC8gZiJ7YXJncy5ydW5fbmFtZX0tZXh0ZXJuYWwtZGF0YSIpLnJlc29sdmUoKQogICAgZm9yIGxhYmVsLCBwYXRoLCBwYXJlbnQgaW4gKAogICAgICAgICgiaW5pdGlhbCBtb2RlbCIsIG1vZGVsX3BhdGgsIHJlcG9fcm9vdCksCiAgICAgICAgKCJydW4gYXJ0aWZhY3RzIiwgcnVuX3Jvb3QsIHJlcG9fcm9vdCAvICJhcnRpZmFjdHMiKSwKICAgICAgICAoInByb2Nlc3NlZCBZT0xPIGRhdGEiLCBleHBvcnRfcm9vdCwgcmVwb19yb290IC8gImRhdGEiIC8gInByb2Nlc3NlZCIpLAogICAgKToKICAgICAgICBpZiBub3QgcGF0aC5pc19yZWxhdGl2ZV90byhwYXJlbnQpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYie2xhYmVsfSBtdXN0IHJlbWFpbiB1bmRlciB7cGFyZW50fToge3BhdGh9IikKICAgIGlmIG5vdCBkYXRhX3Jvb3QuaXNfZGlyKCk6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJDb21wZXRpdGlvbiBpbnB1dCBkaXJlY3RvcnkgZG9lcyBub3QgZXhpc3Q6IHtkYXRhX3Jvb3R9IikKICAgIGthZ2dsZV9pbnB1dF9yb290ID0gUGF0aCgiL2thZ2dsZS9pbnB1dCIpLnJlc29sdmUoKQogICAgaWYgbm90IHN3ZWZpbF9yb290LmlzX3JlbGF0aXZlX3RvKHJlcG9fcm9vdCAvICJkYXRhIiAvICJleHRlcm5hbCIpIGFuZCBub3Qgc3dlZmlsX3Jvb3QuaXNfcmVsYXRpdmVfdG8oa2FnZ2xlX2lucHV0X3Jvb3QpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgICJTV0VGaWwgbXVzdCBiZSBpbnNpZGUgdGhlIHdvcmtzcGFjZSBkYXRhL2V4dGVybmFsIGZvbGRlciBvciBhIHJlYWQtb25seSBLYWdnbGUgaW5wdXQ6ICIKICAgICAgICAgICAgZiJ7c3dlZmlsX3Jvb3R9IgogICAgICAgICkKICAgIGlmIG5vdCBzd2VmaWxfcm9vdC5pc19kaXIoKToKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIlNXRUZpbCBpbnB1dCBkaXJlY3RvcnkgZG9lcyBub3QgZXhpc3Q6IHtzd2VmaWxfcm9vdH0iKQogICAgaWYgbm90IG1vZGVsX3BhdGguaXNfZmlsZSgpOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiSW5pdGlhbCBtb2RlbCBjaGVja3BvaW50IGlzIG1pc3Npbmc6IHttb2RlbF9wYXRofSIpCiAgICBydW5fcm9vdC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBjb25maWcgPSBqc29uLmxvYWRzKChyZXBvX3Jvb3QgLyAiY29uZmlncyIgLyAidmFsaWRhdGlvbi1iYXNlbGluZS5qc29uIikucmVhZF90ZXh0KCkpCiAgICBjb21wZXRpdGlvbiA9IGxvYWRfZGF0YXNldChkYXRhX3Jvb3QpCiAgICBmcm96ZW4gPSBjcmVhdGVfb2JzZXJ2YXRpb25fc3BsaXQoCiAgICAgICAgY29tcGV0aXRpb24sCiAgICAgICAgcmVwb19yb290IC8gImRhdGEiIC8gInNwbGl0cyIsCiAgICAgICAgc2VlZD1pbnQoY29uZmlnWyJzZWVkIl0pLAogICAgICAgIHZhbGlkYXRpb25fZnJhY3Rpb249ZmxvYXQoY29uZmlnWyJ2YWxpZGF0aW9uX2ZyYWN0aW9uIl0pLAogICAgKQogICAgaWYgZnJvemVuLnNoYTI1NiAhPSBjb25maWdbImV4cGVjdGVkX3NwbGl0X3NoYTI1NiJdOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiRnJvemVuIGhvbGRvdXQgZGlnZXN0IGNoYW5nZWQgYmVmb3JlIGV4dGVybmFsIHByZXRyYWluaW5nIikKICAgIGV4dGVybmFsLCBwcm92ZW5hbmNlID0gbG9hZF9zd2VmaWxfZGF0YXNldCgKICAgICAgICBzd2VmaWxfcm9vdCwKICAgICAgICBjb21wZXRpdGlvbiwKICAgICAgICBmcm96ZW4sCiAgICAgICAgcmVwb3NpdG9yeV9yb290PXJlcG9fcm9vdCwKICAgICAgICBleGNsdWRlX2FsbF9jb21wZXRpdGlvbl9kYXRlcz1UcnVlLAogICAgICAgIGFsbG93X3JlYWRvbmx5X2V4dGVybmFsX3Jvb3Q9c3dlZmlsX3Jvb3QuaXNfcmVsYXRpdmVfdG8oa2FnZ2xlX2lucHV0X3Jvb3QpLAogICAgKQogICAgaWYgcHJvdmVuYW5jZVsiaW5jbHVkZWRfb2JzZXJ2YXRpb25zIl0gIT0gMjI5OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgICJFeHBlY3RlZCB0aGUgYXVkaXRlZCAyMjktaW1hZ2UsIGFsbC1jb21wZXRpdGlvbi1kYXRlLWRpc2pvaW50IFNXRUZpbCBzYW1wbGU7ICIKICAgICAgICAgICAgZiJmb3VuZCB7cHJvdmVuYW5jZVsnaW5jbHVkZWRfb2JzZXJ2YXRpb25zJ119IgogICAgICAgICkKICAgIGF1ZGl0X3BhdGggPSBzd2VmaWxfcm9vdCAvICJlcXVpbmV4dXNfYXVkaXRfYWxsX2NvbXBldGl0aW9uX2RhdGVzX2V4Y2x1ZGVkLmpzb24iCiAgICBpZiBub3QgYXVkaXRfcGF0aC5pc19maWxlKCk6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJTdHJpY3QgU1dFRmlsIGF1ZGl0IGlzIG1pc3Npbmc6IHthdWRpdF9wYXRofSIpCiAgICBhdWRpdCA9IGpzb24ubG9hZHMoYXVkaXRfcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICBpZiBhdWRpdC5nZXQoImluY2x1ZGVkX21hbmlmZXN0X3NoYTI1NiIpICE9IHByb3ZlbmFuY2VbImluY2x1ZGVkX21hbmlmZXN0X3NoYTI1NiJdOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlNXRUZpbCBzdHJpY3QgZGF0ZS1kaXNqb2ludCBhdWRpdCBkb2VzIG5vdCBtYXRjaCB0aGUgdHJhaW5pbmcgc2FtcGxlIikKCiAgICBzcGxpdCA9IF9kYXRlX2dyb3VwX3NwbGl0KAogICAgICAgIGV4dGVybmFsLAogICAgICAgIHJlcG9fcm9vdCAvICJkYXRhIiAvICJzcGxpdHMiIC8gZiJ7YXJncy5ydW5fbmFtZX0tZGF0ZS1ncm91cGVkLmNzdiIsCiAgICAgICAgc2VlZD1pbnQoY29uZmlnWyJzZWVkIl0pLAogICAgICAgIHZhbGlkYXRpb25fZnJhY3Rpb249MC4yLAogICAgKQogICAgZXhwb3J0X21hbmlmZXN0ID0gZXhwb3J0X3lvbG9fZGF0YXNldChleHRlcm5hbCwgc3BsaXQsIGV4cG9ydF9yb290KQogICAgZXhwb3J0X21hbmlmZXN0LnBvcCgidmlld3MiLCBOb25lKQogICAgKHJ1bl9yb290IC8gImV4dGVybmFsLWV4cG9ydC1tYW5pZmVzdC5qc29uIikud3JpdGVfdGV4dCgKICAgICAgICBqc29uLmR1bXBzKGV4cG9ydF9tYW5pZmVzdCwgaW5kZW50PTIsIHNvcnRfa2V5cz1UcnVlKSArICJcbiIsIGVuY29kaW5nPSJ1dGYtOCIKICAgICkKICAgIHByb3ZlbmFuY2UudXBkYXRlKAogICAgICAgIHsKICAgICAgICAgICAgImNvbXBldGl0aW9uX2hvbGRvdXRfc2hhMjU2IjogZnJvemVuLnNoYTI1NiwKICAgICAgICAgICAgIm9vZl9zYWZlX2RhdGVfZXhjbHVzaW9uIjogVHJ1ZSwKICAgICAgICAgICAgImV4dGVybmFsX3RyYWluX3ZhbGlkYXRpb25fc3BsaXQiOiB7CiAgICAgICAgICAgICAgICAibWFuaWZlc3QiOiBzdHIoc3BsaXQubWFuaWZlc3RfcGF0aC5yZWxhdGl2ZV90byhyZXBvX3Jvb3QpKSwKICAgICAgICAgICAgICAgICJzaGEyNTYiOiBzcGxpdC5zaGEyNTYsCiAgICAgICAgICAgICAgICAiZ3JvdXBfa2V5IjogIlVUQyBjYWxlbmRhciBkYXRlIiwKICAgICAgICAgICAgICAgICJ0cmFpbl9vYnNlcnZhdGlvbnMiOiBsZW4oc3BsaXQudHJhaW5pbmdfbmFtZXMpLAogICAgICAgICAgICAgICAgInZhbGlkYXRpb25fb2JzZXJ2YXRpb25zIjogbGVuKHNwbGl0LnZhbGlkYXRpb25fbmFtZXMpLAogICAgICAgICAgICAgICAgInZhbGlkYXRpb25fZGF5cyI6IGxlbigKICAgICAgICAgICAgICAgICAgICB7UGF0aChuYW1lKS5zdGVtWzo4XSBmb3IgbmFtZSBpbiBzcGxpdC52YWxpZGF0aW9uX25hbWVzfQogICAgICAgICAgICAgICAgKSwKICAgICAgICAgICAgfSwKICAgICAgICAgICAgImV4cG9ydCI6IGV4cG9ydF9tYW5pZmVzdCwKICAgICAgICB9CiAgICApCiAgICAocnVuX3Jvb3QgLyAiZXh0ZXJuYWwtZGF0YS1wcm92ZW5hbmNlLmpzb24iKS53cml0ZV90ZXh0KAogICAgICAgIGpzb24uZHVtcHMocHJvdmVuYW5jZSwgaW5kZW50PTIsIHNvcnRfa2V5cz1UcnVlKSArICJcbiIsIGVuY29kaW5nPSJ1dGYtOCIKICAgICkKCiAgICBtb2RlbF9jYWNoZSA9IHJ1bl9yb290IC8gInVsdHJhbHl0aWNzLWNvbmZpZyIKICAgIG1vZGVsX2NhY2hlLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIG9zLmVudmlyb25bIllPTE9fQ09ORklHX0RJUiJdID0gc3RyKG1vZGVsX2NhY2hlKQogICAgb3MuZW52aXJvblsiVE9SQ0hfSE9NRSJdID0gc3RyKHJ1bl9yb290IC8gInRvcmNoLWhvbWUiKQoKICAgIGltcG9ydCB0b3JjaAogICAgaW1wb3J0IHVsdHJhbHl0aWNzCiAgICBmcm9tIHVsdHJhbHl0aWNzIGltcG9ydCBZT0xPCgogICAgaWYgbm90IHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgYW5kIGFyZ3MuZGV2aWNlICE9ICJjcHUiOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiQ1VEQSBpcyB1bmF2YWlsYWJsZTsgdXNlIC0tZGV2aWNlIGNwdSBvbmx5IGZvciBhIHNtb2tlIHJ1biIpCiAgICByYW5kb20uc2VlZChpbnQoY29uZmlnWyJzZWVkIl0pKQogICAgbnAucmFuZG9tLnNlZWQoaW50KGNvbmZpZ1sic2VlZCJdKSkKICAgIHRvcmNoLm1hbnVhbF9zZWVkKGludChjb25maWdbInNlZWQiXSkpCiAgICBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgIHRvcmNoLmN1ZGEubWFudWFsX3NlZWRfYWxsKGludChjb25maWdbInNlZWQiXSkpCiAgICAgICAgdG9yY2guY3VkYS5yZXNldF9wZWFrX21lbW9yeV9zdGF0cygpCiAgICBtb2RlbCA9IFlPTE8oc3RyKG1vZGVsX3BhdGgpKQogICAgc3RhcnRlZCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgIG1vZGVsLnRyYWluKAogICAgICAgIGRhdGE9c3RyKGV4cG9ydF9yb290IC8gImRhdGEueWFtbCIpLAogICAgICAgIGltZ3N6PWFyZ3MuaW1hZ2Vfc2l6ZSwKICAgICAgICBlcG9jaHM9YXJncy5lcG9jaHMsCiAgICAgICAgYmF0Y2g9YXJncy5iYXRjaF9zaXplLAogICAgICAgIGRldmljZT1hcmdzLmRldmljZSwKICAgICAgICB3b3JrZXJzPWFyZ3Mud29ya2VycywKICAgICAgICBwYXRpZW5jZT1hcmdzLnBhdGllbmNlLAogICAgICAgIHNlZWQ9aW50KGNvbmZpZ1sic2VlZCJdKSwKICAgICAgICBkZXRlcm1pbmlzdGljPUZhbHNlLAogICAgICAgIHNpbmdsZV9jbHM9VHJ1ZSwKICAgICAgICBjb3NfbHI9VHJ1ZSwKICAgICAgICBwbG90cz1GYWxzZSwKICAgICAgICBtb3NhaWM9MC4wLAogICAgICAgIGNsb3NlX21vc2FpYz0wLAogICAgICAgIG1peHVwPTAuMCwKICAgICAgICBjb3B5X3Bhc3RlPTAuMCwKICAgICAgICBmbGlwbHI9MC41LAogICAgICAgIGZsaXB1ZD0wLjUsCiAgICAgICAgZGVncmVlcz0wLjAsCiAgICAgICAgc2hlYXI9MC4wLAogICAgICAgIHBlcnNwZWN0aXZlPTAuMCwKICAgICAgICB0cmFuc2xhdGU9MC4wNSwKICAgICAgICBzY2FsZT0wLjIsCiAgICAgICAgaHN2X2g9MC4wLAogICAgICAgIGhzdl9zPTAuMCwKICAgICAgICBoc3Zfdj0wLjIsCiAgICAgICAgb3ZlcmxhcF9tYXNrPVRydWUsCiAgICAgICAgbWFza19yYXRpbz0yLAogICAgICAgIHByb2plY3Q9c3RyKHJ1bl9yb290IC8gInRyYWluIiksCiAgICAgICAgbmFtZT0ieW9sby1zZWciLAogICAgICAgIGV4aXN0X29rPVRydWUsCiAgICAgICAgc2F2ZT1UcnVlLAogICAgICAgIHNhdmVfcGVyaW9kPW1heCgxLCBhcmdzLmVwb2NocyAvLyAxMCksCiAgICAgICAgYW1wPVRydWUsCiAgICAgICAgY2FjaGU9RmFsc2UsCiAgICAgICAgdmVyYm9zZT1UcnVlLAogICAgKQogICAgZWxhcHNlZCA9IHRpbWUucGVyZl9jb3VudGVyKCkgLSBzdGFydGVkCiAgICBiZXN0ID0gcnVuX3Jvb3QgLyAidHJhaW4iIC8gInlvbG8tc2VnIiAvICJ3ZWlnaHRzIiAvICJiZXN0LnB0IgogICAgbGFzdCA9IHJ1bl9yb290IC8gInRyYWluIiAvICJ5b2xvLXNlZyIgLyAid2VpZ2h0cyIgLyAibGFzdC5wdCIKICAgIGlmIG5vdCBiZXN0LmlzX2ZpbGUoKToKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIllPTE8gZGlkIG5vdCBzYXZlIHRoZSBiZXN0IGNoZWNrcG9pbnQ6IHtiZXN0fSIpCiAgICBpbXBvcnQgc2h1dGlsCgogICAgc2h1dGlsLmNvcHkyKGJlc3QsIHJ1bl9yb290IC8gImJlc3QucHQiKQogICAgaWYgbGFzdC5pc19maWxlKCk6CiAgICAgICAgc2h1dGlsLmNvcHkyKGxhc3QsIHJ1bl9yb290IC8gImxhc3QucHQiKQogICAgc291cmNlX2NvbW1pdCA9IG9zLmVudmlyb24uZ2V0KCJFUVVJTkVYVVNfU09VUkNFX0NPTU1JVCIpCiAgICBpZiBzb3VyY2VfY29tbWl0IGlzIE5vbmU6CiAgICAgICAgc291cmNlX2NvbW1pdCA9IF9faW1wb3J0X18oInN1YnByb2Nlc3MiKS5ydW4oCiAgICAgICAgICAgIFsiZ2l0IiwgIi1DIiwgc3RyKHJlcG9fcm9vdCksICJyZXYtcGFyc2UiLCAiSEVBRCJdLAogICAgICAgICAgICBjaGVjaz1UcnVlLAogICAgICAgICAgICBjYXB0dXJlX291dHB1dD1UcnVlLAogICAgICAgICAgICB0ZXh0PVRydWUsCiAgICAgICAgKS5zdGRvdXQuc3RyaXAoKQogICAgcmVzdWx0ID0gewogICAgICAgICJzdGF0dXMiOiAiY29tcGxldGUiLAogICAgICAgICJydW5fbmFtZSI6IGFyZ3MucnVuX25hbWUsCiAgICAgICAgInNvdXJjZV9jb21taXQiOiBzb3VyY2VfY29tbWl0LAogICAgICAgICJzb3VyY2VfYnVuZGxlX3NoYTI1NiI6IG9zLmVudmlyb24uZ2V0KCJFUVVJTkVYVVNfU09VUkNFX0JVTkRMRV9TSEEyNTYiKSwKICAgICAgICAiZnJvemVuX2hvbGRvdXRfc2hhMjU2IjogZnJvemVuLnNoYTI1NiwKICAgICAgICAiZnJvemVuX2hvbGRvdXRfdXNlZF9mb3JfdHJhaW5pbmdfb3Jfc2VsZWN0aW9uIjogRmFsc2UsCiAgICAgICAgInRyYWluaW5nX2RhdGEiOiAiU1dFRmlsIENDLUJZLTQuMCBleHRlcm5hbCBhbm5vdGF0aW9ucyBvbmx5IiwKICAgICAgICAidHJhaW5pbmdfb2JzZXJ2YXRpb25zIjogbGVuKHNwbGl0LnRyYWluaW5nX25hbWVzKSwKICAgICAgICAidmFsaWRhdGlvbl9vYnNlcnZhdGlvbnMiOiBsZW4oc3BsaXQudmFsaWRhdGlvbl9uYW1lcyksCiAgICAgICAgImV4dGVybmFsX2RhdGFfcHJvdmVuYW5jZSI6IHN0cigocnVuX3Jvb3QgLyAiZXh0ZXJuYWwtZGF0YS1wcm92ZW5hbmNlLmpzb24iKS5yZWxhdGl2ZV90byhyZXBvX3Jvb3QpKSwKICAgICAgICAiaW5pdGlhbF9tb2RlbCI6IHsKICAgICAgICAgICAgIm5hbWUiOiBtb2RlbF9wYXRoLm5hbWUsCiAgICAgICAgICAgICJzb3VyY2UiOiAiVWx0cmFseXRpY3MgWU9MTzExLXNlZyBDT0NPIHByZXRyYWluZWQgd2VpZ2h0cyIsCiAgICAgICAgICAgICJsaWNlbnNlIjogIkFHUEwtMy4wIiwKICAgICAgICAgICAgImNoZWNrcG9pbnRfc2hhMjU2IjogX3NoYTI1Nihtb2RlbF9wYXRoKSwKICAgICAgICB9LAogICAgICAgICJjb25maWd1cmF0aW9uIjogewogICAgICAgICAgICAiaW1hZ2Vfc2l6ZSI6IGFyZ3MuaW1hZ2Vfc2l6ZSwKICAgICAgICAgICAgImVwb2NocyI6IGFyZ3MuZXBvY2hzLAogICAgICAgICAgICAiYmF0Y2hfc2l6ZSI6IGFyZ3MuYmF0Y2hfc2l6ZSwKICAgICAgICAgICAgInBhdGllbmNlIjogYXJncy5wYXRpZW5jZSwKICAgICAgICAgICAgIndvcmtlcnMiOiBhcmdzLndvcmtlcnMsCiAgICAgICAgICAgICJkZXZpY2UiOiBhcmdzLmRldmljZSwKICAgICAgICAgICAgInNlZWQiOiBpbnQoY29uZmlnWyJzZWVkIl0pLAogICAgICAgICAgICAiYXVnbWVudGF0aW9uIjogImZsaXBzLCBtaWxkIHRyYW5zbGF0aW9uL3NjYWxlL2JyaWdodG5lc3M7IG1vc2FpYy9taXh1cCBkaXNhYmxlZCIsCiAgICAgICAgfSwKICAgICAgICAiY2hlY2twb2ludCI6IHN0cigocnVuX3Jvb3QgLyAiYmVzdC5wdCIpLnJlbGF0aXZlX3RvKHJlcG9fcm9vdCkpLAogICAgICAgICJiZXN0X2NoZWNrcG9pbnRfc2hhMjU2IjogX3NoYTI1NihydW5fcm9vdCAvICJiZXN0LnB0IiksCiAgICAgICAgInRyYWluaW5nX3NlY29uZHMiOiBlbGFwc2VkLAogICAgICAgICJwZWFrX2dwdV9tZW1vcnlfYnl0ZXMiOiBpbnQodG9yY2guY3VkYS5tYXhfbWVtb3J5X2FsbG9jYXRlZCgpKSBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgMCwKICAgICAgICAiaGFyZHdhcmUiOiBoYXJkd2FyZV9pbmZvKCksCiAgICAgICAgInBhY2thZ2VfdmVyc2lvbnMiOiB7KipwYWNrYWdlX3ZlcnNpb25zKCksICJ1bHRyYWx5dGljcyI6IHVsdHJhbHl0aWNzLl9fdmVyc2lvbl9ffSwKICAgIH0KICAgIChydW5fcm9vdCAvICJydW4tbWV0YWRhdGEuanNvbiIpLndyaXRlX3RleHQoCiAgICAgICAganNvbi5kdW1wcyhyZXN1bHQsIGluZGVudD0yLCBzb3J0X2tleXM9VHJ1ZSkgKyAiXG4iLCBlbmNvZGluZz0idXRmLTgiCiAgICApCiAgICBwcmludChqc29uLmR1bXBzKHJlc3VsdCwgaW5kZW50PTIsIHNvcnRfa2V5cz1UcnVlKSwgZmx1c2g9VHJ1ZSkKICAgIHJldHVybiByZXN1bHQKCgpkZWYgbWFpbigpIC0+IGludDoKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKAogICAgICAgIGRlc2NyaXB0aW9uPSJQcmV0cmFpbiBZT0xPMTEtc2VnIG9uIGRhdGUtZGlzam9pbnQsIHNlcGFyYXRlbHkgbGljZW5zZWQgU1dFRmlsIG1hc2tzLiIKICAgICkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcmVwby1yb290IiwgdHlwZT1QYXRoLCBkZWZhdWx0PVJFUE9fUk9PVCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGF0YS1yb290IiwgdHlwZT1QYXRoLCBkZWZhdWx0PU5vbmUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXN3ZWZpbC1yb290IiwgdHlwZT1QYXRoLCBkZWZhdWx0PU5vbmUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1vZGVsIiwgdHlwZT1QYXRoLCBkZWZhdWx0PVJFUE9fUk9PVCAvICJ3ZWlnaHRzIiAvICJ5b2xvMTFzLXNlZy5wdCIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXJ1bi1uYW1lIiwgZGVmYXVsdD0ieW9sbzExcy1zZWctc3dlZmlsLXByZXRyYWluIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0taW1hZ2Utc2l6ZSIsIHR5cGU9aW50LCBkZWZhdWx0PTEwMjQpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWVwb2NocyIsIHR5cGU9aW50LCBkZWZhdWx0PTMwKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1iYXRjaC1zaXplIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0td29ya2VycyIsIHR5cGU9aW50LCBkZWZhdWx0PTQpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXBhdGllbmNlIiwgdHlwZT1pbnQsIGRlZmF1bHQ9OCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGV2aWNlIiwgZGVmYXVsdD0iMCIpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgcnVuKGFyZ3MpCiAgICByZXR1cm4gMAoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICByYWlzZSBTeXN0ZW1FeGl0KG1haW4oKSkK","configs/validation-baseline.json":"ewogICJzZWVkIjogNDIsCiAgInZhbGlkYXRpb25fZnJhY3Rpb24iOiAwLjIsCiAgInN0cmF0aWZ5X2J5IjogInllYXIiLAogICJleHBlY3RlZF9zcGxpdF9zaGEyNTYiOiAiMzBjYjJlYzUyNDM3ZGFmOTdlNzY5YTNhNjFiZjgzMzUwMDYyZTZjZjRmNWQwMGFmMWY4Mzk5NjQ5MDY1ZTQ4ZSIsCiAgImJsdXJfa2VybmVsIjogMzEsCiAgInNvbGFyX2Rpc2tfcmFkaXVzX2ZyYWN0aW9uIjogMC40NCwKICAiY2xvc2luZ19rZXJuZWwiOiAzLAogICJjb21wb25lbnRfY29ubmVjdGl2aXR5IjogOCwKICAibWluX2NvbXBvbmVudF9hcmVhIjogNTAsCiAgIm1heF9jb21wb25lbnRfYXJlYSI6IDUwMDAwLAogICJpb3VfdGhyZXNob2xkIjogMC41LAogICJkaWFnbm9zdGljX292ZXJsYXBfdGhyZXNob2xkIjogMC4wCn0K","requirements-detector.txt":"IyBPcHRpb25hbCBpbnN0YW5jZS1kZXRlY3RvciBlbnZpcm9ubWVudCBmb3IgdGhlIEthZ2dsZSBHUFUgc3ByaW50LgojIFVsdHJhbHl0aWNzIFlPTE8xMSBjb2RlIGFuZCBwcmV0cmFpbmVkIENPQ08gd2VpZ2h0cyBhcmUgZGlzdHJpYnV0ZWQgdW5kZXIgQUdQTC0zLjAuCnVsdHJhbHl0aWNzPT04LjQuMTczCg=="}
for relative, payload in SOURCE_FILES.items():
    destination = REPOSITORY_PATH / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(base64.b64decode(payload))
print(f'Embedded {len(SOURCE_FILES)} source files from {SOURCE_COMMIT}')
print(f'Embedded source SHA-256: {SOURCE_BUNDLE_SHA256}')


In [ ]:
import os, subprocess, sys
os.environ['EQUINEXUS_SOURCE_COMMIT'] = SOURCE_COMMIT
os.environ['EQUINEXUS_SOURCE_BUNDLE_SHA256'] = SOURCE_BUNDLE_SHA256
os.environ['YOLO_CONFIG_DIR'] = '/kaggle/working/equinexus/yolo-config'
os.environ['TORCH_HOME'] = '/kaggle/working/equinexus/torch-home'
for key in ('YOLO_CONFIG_DIR', 'TORCH_HOME'):
    Path(os.environ[key]).mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--no-cache-dir', '-r', str(REPOSITORY_PATH / 'requirements-detector.txt')], check=True)
import torch, ultralytics
devices = [{'name': torch.cuda.get_device_name(i), 'total_memory_bytes': torch.cuda.get_device_properties(i).total_memory} for i in range(torch.cuda.device_count())]
print({'python': sys.version.split()[0], 'torch': torch.__version__, 'cuda': torch.version.cuda, 'ultralytics': ultralytics.__version__, 'cuda_available': torch.cuda.is_available(), 'devices': devices})
assert torch.cuda.is_available(), 'A Kaggle GPU is required for this run'
input_root = Path('/kaggle/input')
revision_markers = sorted(input_root.rglob('equinexus_revision.txt'))
if not revision_markers:
    mounted_inputs = sorted(str(path) for path in input_root.iterdir()) if input_root.is_dir() else []
    raise FileNotFoundError(f'No SWEFil revision marker found under {input_root}; mounted inputs: {mounted_inputs}')
if len(revision_markers) != 1:
    raise RuntimeError(f'Expected one attached SWEFil dataset, found revision markers: {revision_markers}')
swefil_root = revision_markers[0].parent
print('Mounted SWEFil dataset:', swefil_root)
assert (swefil_root / 'equinexus_revision.txt').read_text().strip() == 'bc818c69a79266ad0fa91a539359485af57967cf', 'Unexpected SWEFil revision'
weights_dir = REPOSITORY_PATH / 'weights'
weights_dir.mkdir(parents=True, exist_ok=True)
from ultralytics import YOLO
YOLO('yolo11s-seg.pt')
import shutil
downloaded = Path('yolo11s-seg.pt').resolve()
if not downloaded.is_file():
    downloaded = Path.home() / '.config' / 'Ultralytics' / 'yolo11s-seg.pt'
assert downloaded.is_file(), f'Could not locate downloaded COCO checkpoint: {downloaded}'
shutil.copy2(downloaded, weights_dir / 'yolo11s-seg.pt')
print('Prepared SWEFil data and COCO initialization checkpoint')


In [ ]:
command = [sys.executable, '-u', str(REPOSITORY_PATH / 'scripts' / 'run_yolo_swefil_pretrain.py'), '--repo-root', str(REPOSITORY_PATH), '--data-root', '/kaggle/input', '--swefil-root', str(swefil_root), '--model', str(weights_dir / 'yolo11s-seg.pt'), '--run-name', 'yolo11s-seg-swefil-30ep', '--image-size', '1024', '--epochs', '30', '--batch-size', '4', '--workers', '4', '--patience', '8', '--device', '0']
print('Starting external YOLO pretraining:', ' '.join(command), flush=True)
subprocess.run(command, check=True)
import json
metadata_path = REPOSITORY_PATH / 'artifacts' / 'detector-experiments' / 'yolo11s-seg-swefil-30ep' / 'run-metadata.json'
print(json.dumps(json.loads(metadata_path.read_text()), indent=2))
print('Best checkpoint:', metadata_path.parent / 'best.pt')
